# Agent-Based Muay Thai Individual Strike Classifier

This notebook evaluates the LangGraph agent approach (from `individual_agent/basic_agent_explicit.py`) to serve as a comparison for the DSPy and baseline classifiers on individual strikes. It uses the same data splits and evaluation metrics.


## Section 1: Setup and Configuration


In [1]:
import json
from pathlib import Path
import os
from dotenv import load_dotenv
import random
import time
from typing import List, Dict, Tuple
import re
from collections import defaultdict

# Import agent from basic_agent_explicit
import sys
sys.path.append('individual_agent')  # Add agent directory to path
from basic_agent_explicit import graph, llm
from langchain_core.messages import HumanMessage, ToolMessage, AIMessage

# Load environment variables
load_dotenv()

# Data paths (same as other notebooks)
DATA_ROOT = Path("data/individual_strikes")

# Training split size (5 examples per strike type for training, same as baseline evaluator)
TRAIN_SIZE = 5

# Rate limiting for free tier
# Agent makes multiple API calls (analyze_video + potentially extract_pose_data)
# Use 60 seconds between videos to avoid rate limits
REQUEST_DELAY = 60

print("Setup complete!")
print(f"Data directory: {DATA_ROOT}")
print(f"Rate limiting: {REQUEST_DELAY}s delay between videos")
print(f"Agent graph loaded successfully")


Setup complete!
Data directory: data\individual_strikes
Rate limiting: 60s delay between videos
Agent graph loaded successfully


## Section 2: Data Loading


In [2]:
def load_individual_strike_examples(data_root: Path) -> List[Dict]:
    """
    Load all individual strike videos from subdirectories.
    Each directory contains a label.txt file with the strike type.
    Returns a list of examples with video_path and strike_type.
    """
    examples = []
    
    # Iterate through all subdirectories in data/individual_strikes
    for strike_dir in sorted(data_root.iterdir()):
        if not strike_dir.is_dir():
            continue
        
        # Read strike type from label.txt file
        label_file = strike_dir / "label.txt"
        if not label_file.exists():
            print(f"Warning: label.txt not found in {strike_dir.name}, skipping...")
            continue
        
        try:
            with open(label_file, 'r') as f:
                strike_type = f.read().strip()
            
            if not strike_type:
                print(f"Warning: label.txt is empty in {strike_dir.name}, skipping...")
                continue
        except Exception as e:
            print(f"Warning: Error reading label.txt in {strike_dir.name}: {e}, skipping...")
            continue
        
        # Get all .mov files in this directory
        video_files = sorted(strike_dir.glob("*.mov"))
        
        for video_path in video_files:
            example = {
                'video_path': str(video_path),
                'strike_type': strike_type  # Label comes from label.txt file
            }
            examples.append(example)
    
    return examples

# Load all examples
all_examples = load_individual_strike_examples(DATA_ROOT)

print(f"Loaded {len(all_examples)} total examples")

# Group examples by strike type
examples_by_strike = {}
for example in all_examples:
    strike_type = example['strike_type']
    if strike_type not in examples_by_strike:
        examples_by_strike[strike_type] = []
    examples_by_strike[strike_type].append(example)

print(f"\nExamples per strike type:")
for strike_type in sorted(examples_by_strike.keys()):
    count = len(examples_by_strike[strike_type])
    print(f"  {strike_type}: {count} videos")


Loaded 336 total examples

Examples per strike type:
  cross: 24 videos
  jab: 24 videos
  knee: 24 videos
  lead_elbow: 24 videos
  lead_hook: 24 videos
  lead_uppercut: 24 videos
  low_kick: 24 videos
  rear_elbow: 24 videos
  rear_hook: 24 videos
  rear_uppercut: 24 videos
  roundhouse: 24 videos
  switch_kick: 24 videos
  switch_knee: 24 videos
  teep: 24 videos


In [3]:
# Split data into train and validation
# Use the same random seed (42) as DSPy and baseline notebooks for reproducibility
# Training set: 5 examples per strike type, rest for validation

random.seed(42)  # Same seed as other notebooks

train_examples = []
val_examples = []

for strike_type, strike_examples in sorted(examples_by_strike.items()):
    # Shuffle examples for this strike type
    indices = list(range(len(strike_examples)))
    random.shuffle(indices)
    
    # Select first TRAIN_SIZE for training
    train_indices = set(indices[:TRAIN_SIZE])
    train_set = [strike_examples[i] for i in train_indices]
    val_set = [strike_examples[i] for i in range(len(strike_examples)) if i not in train_indices]
    
    train_examples.extend(train_set)
    val_examples.extend(val_set)
    
    print(f"{strike_type}: {len(train_set)} train, {len(val_set)} val")

print(f"\nTraining set: {len(train_examples)} examples")
print(f"Validation set: {len(val_examples)} examples")

# Display statistics
unique_strikes = set(ex['strike_type'] for ex in train_examples + val_examples)
print(f"\nUnique strike types: {sorted(unique_strikes)}")
print(f"Total unique strike types: {len(unique_strikes)}")


cross: 5 train, 19 val
jab: 5 train, 19 val
knee: 5 train, 19 val
lead_elbow: 5 train, 19 val
lead_hook: 5 train, 19 val
lead_uppercut: 5 train, 19 val
low_kick: 5 train, 19 val
rear_elbow: 5 train, 19 val
rear_hook: 5 train, 19 val
rear_uppercut: 5 train, 19 val
roundhouse: 5 train, 19 val
switch_kick: 5 train, 19 val
switch_knee: 5 train, 19 val
teep: 5 train, 19 val

Training set: 70 examples
Validation set: 266 examples

Unique strike types: ['cross', 'jab', 'knee', 'lead_elbow', 'lead_hook', 'lead_uppercut', 'low_kick', 'rear_elbow', 'rear_hook', 'rear_uppercut', 'roundhouse', 'switch_kick', 'switch_knee', 'teep']
Total unique strike types: 14


## Section 3: Agent Classification Function


In [4]:
def normalize_strike_name(strike: str) -> str:
    """
    Normalize strike name to match directory naming convention.
    Handles variations like spaces, hyphens, case differences.
    """
    # Convert to lowercase and replace spaces/hyphens with underscores
    normalized = strike.lower().strip()
    normalized = normalized.replace(' ', '_').replace('-', '_')
    return normalized

def count_tool_calls(messages) -> Dict[str, int]:
    """
    Count tool calls from agent message history.
    Returns a dictionary with tool name as key and count as value.
    """
    tool_counts = {'analyze_video': 0, 'extract_pose_data': 0}
    
    for message in messages:
        # Check for AIMessage with tool_calls
        if isinstance(message, AIMessage):
            # Check if message has tool_calls attribute
            if hasattr(message, 'tool_calls') and message.tool_calls:
                for tool_call in message.tool_calls:
                    # tool_call can be a dict or an object with name attribute
                    if isinstance(tool_call, dict):
                        tool_name = tool_call.get('name', '')
                    elif hasattr(tool_call, 'name'):
                        tool_name = tool_call.name
                    else:
                        tool_name = str(tool_call)
                    
                    # Normalize tool name (remove any prefixes/suffixes)
                    tool_name = tool_name.strip()
                    if tool_name in tool_counts:
                        tool_counts[tool_name] += 1
        
        # Check for ToolMessage (tool execution results)
        elif isinstance(message, ToolMessage):
            # ToolMessage has a 'name' attribute that indicates which tool was called
            if hasattr(message, 'name'):
                tool_name = message.name
                if tool_name in tool_counts:
                    tool_counts[tool_name] += 1
    
    return tool_counts

def classify_video_agent(video_path: str) -> Tuple[str, Dict[str, int]]:
    """
    Classify the single Muay Thai strike in a video using the LangGraph agent.
    Returns a tuple of (predicted_strike, tool_usage_counts).
    """
    tool_usage = {'analyze_video': 0, 'extract_pose_data': 0}
    
    try:
        # Invoke the agent
        result = graph.invoke(
            {"messages": [HumanMessage(content=f"Classify the strike in this video: {video_path}")]}
        )
        
        # Count tool calls from message history
        tool_usage = count_tool_calls(result["messages"])
        
        # Extract response from the last message
        agent_response = result["messages"][-1].content
        
        # Handle different response formats
        # Response might be a string, list, or dict
        if isinstance(agent_response, str):
            response_text = agent_response
        elif isinstance(agent_response, list):
            # If response is a list of content blocks, extract text
            response_text = ""
            for item in agent_response:
                if isinstance(item, dict) and 'text' in item:
                    response_text += item['text']
                elif isinstance(item, str):
                    response_text += item
        elif isinstance(agent_response, dict):
            # If response is a dict, try to extract text
            response_text = agent_response.get('text', str(agent_response))
        else:
            response_text = str(agent_response)
        
        # Clean up the response - remove quotes, brackets, etc.
        response_text = response_text.strip()
        response_text = response_text.strip('"\'[]')
        
        # Normalize the strike name
        predicted_strike = normalize_strike_name(response_text)
        
        return predicted_strike, tool_usage
        
    except Exception as e:
        print(f"Error classifying video {video_path}: {e}")
        return "", tool_usage

print("Agent classification function created!")


Agent classification function created!


## Section 4: Evaluation


In [5]:
# Evaluation metrics (same as baseline evaluator for individual strikes)

def exact_match_metric(true_strike: str, predicted_strike: str) -> bool:
    """Evaluate exact match between predicted and true strike."""
    return normalize_strike_name(true_strike) == normalize_strike_name(predicted_strike)

print("Evaluation metrics defined!")


Evaluation metrics defined!


In [6]:
# Evaluate on validation set
print(f"Evaluating on {len(val_examples)} validation examples...")
print(f"Using {REQUEST_DELAY}s delay between videos to avoid rate limits")
print(f"(Agent makes multiple API calls per video, so longer delays are needed)\n")

results = []
exact_matches = 0
total_tool_usage = {'analyze_video': 0, 'extract_pose_data': 0}

for i, example in enumerate(val_examples):
    try:
        # Add rate limiting delay (except for first request)
        if i > 0:
            print(f"\nWaiting {REQUEST_DELAY}s to avoid rate limits...")
            time.sleep(REQUEST_DELAY)
        
        print(f"\n{'='*60}")
        print(f"Processing {i+1}/{len(val_examples)}: {Path(example['video_path']).name}")
        print(f"{'='*60}")
        
        # Classify video using agent
        predicted_strike, tool_usage = classify_video_agent(example['video_path'])
        
        # Track tool usage
        total_tool_usage['analyze_video'] += tool_usage['analyze_video']
        total_tool_usage['extract_pose_data'] += tool_usage['extract_pose_data']
        
        true_strike = example['strike_type']
        exact_match = exact_match_metric(true_strike, predicted_strike)
        
        if exact_match:
            exact_matches += 1
        
        results.append({
            'video_path': example['video_path'],
            'true_strike': true_strike,
            'predicted_strike': predicted_strike,
            'exact_match': exact_match,
            'tool_usage': tool_usage
        })
        
        print(f"True:  {true_strike}")
        print(f"Pred:  {predicted_strike}")
        print(f"Match: {'✓' if exact_match else '✗'}")
        print(f"Tools: analyze_video={tool_usage['analyze_video']}, extract_pose_data={tool_usage['extract_pose_data']}")
            
    except Exception as e:
        print(f"Error processing {example['video_path']}: {e}\n")
        results.append({
            'video_path': example['video_path'],
            'true_strike': example['strike_type'],
            'predicted_strike': '',
            'exact_match': False,
            'tool_usage': {'analyze_video': 0, 'extract_pose_data': 0},
            'error': str(e)
        })

print("\n" + "="*60)
print("Evaluation complete!")
print("="*60)


Evaluating on 266 validation examples...
Using 60s delay between videos to avoid rate limits
(Agent makes multiple API calls per video, so longer delays are needed)


Processing 1/266: 1.mov
[DEBUG] analyze_video returned: cross...
True:  cross
Pred:  cross
Match: ✓
Tools: analyze_video=2, extract_pose_data=2

Waiting 60s to avoid rate limits...

Processing 2/266: 10.mov
[DEBUG] analyze_video returned: jab...
True:  cross
Pred:  jab
Match: ✗
Tools: analyze_video=2, extract_pose_data=2

Waiting 60s to avoid rate limits...

Processing 3/266: 11.mov
[DEBUG] analyze_video returned: teep...
True:  cross
Pred:  teep
Match: ✗
Tools: analyze_video=2, extract_pose_data=2

Waiting 60s to avoid rate limits...

Processing 4/266: 12.mov
[DEBUG] analyze_video returned: teep...
True:  cross
Pred:  teep
Match: ✗
Tools: analyze_video=2, extract_pose_data=2

Waiting 60s to avoid rate limits...

Processing 5/266: 13.mov
[DEBUG] analyze_video returned: teep...
True:  cross
Pred:  teep
Match: ✗
Tools: anal

In [7]:
# Calculate and display metrics
exact_match_accuracy = exact_matches / len(val_examples) if val_examples else 0

print("=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)
print(f"Exact Match Accuracy: {exact_match_accuracy:.2%} ({exact_matches}/{len(val_examples)})")
print()

# Display some example predictions
print("Sample Predictions:")
print("-" * 60)
for i, result in enumerate(results[:5], 1):
    print(f"\nExample {i}:")
    print(f"  Video: {Path(result['video_path']).name}")
    print(f"  True:  {result['true_strike']}")
    print(f"  Pred:  {result['predicted_strike']}")
    print(f"  Match: {'✓' if result['exact_match'] else '✗'}")


EVALUATION RESULTS
Exact Match Accuracy: 14.66% (39/266)

Sample Predictions:
------------------------------------------------------------

Example 1:
  Video: 1.mov
  True:  cross
  Pred:  cross
  Match: ✓

Example 2:
  Video: 10.mov
  True:  cross
  Pred:  jab
  Match: ✗

Example 3:
  Video: 11.mov
  True:  cross
  Pred:  teep
  Match: ✗

Example 4:
  Video: 12.mov
  True:  cross
  Pred:  teep
  Match: ✗

Example 5:
  Video: 13.mov
  True:  cross
  Pred:  teep
  Match: ✗


In [8]:
# Calculate per-strike accuracy
strike_stats = defaultdict(lambda: {'correct': 0, 'total': 0, 'predicted': 0})

for result in results:
    true_strike = result['true_strike']
    pred_strike = result['predicted_strike']
    
    # Count true strikes
    strike_stats[true_strike]['total'] += 1
    
    # Count predicted strikes
    if pred_strike:
        strike_stats[pred_strike]['predicted'] += 1
    
    # Count correct predictions
    if result['exact_match']:
        strike_stats[true_strike]['correct'] += 1

print("\nPer-Strike Accuracy:")
print("-" * 60)
for strike in sorted(strike_stats.keys()):
    stats = strike_stats[strike]
    accuracy = stats['correct'] / stats['total'] if stats['total'] > 0 else 0
    print(f"  {strike:20s}: {accuracy:6.2%} ({stats['correct']}/{stats['total']} correct, {stats['predicted']} predicted)")

# Display tool usage statistics
print("\n" + "=" * 60)
print("TOOL USAGE STATISTICS")
print("=" * 60)
print(f"Total tool calls across all videos:")
print(f"  analyze_video:      {total_tool_usage['analyze_video']} calls")
print(f"  extract_pose_data:  {total_tool_usage['extract_pose_data']} calls")
print(f"  Total:               {sum(total_tool_usage.values())} calls")
print(f"\nAverage tool calls per video:")
print(f"  analyze_video:      {total_tool_usage['analyze_video'] / len(val_examples):.2f} calls/video")
print(f"  extract_pose_data:  {total_tool_usage['extract_pose_data'] / len(val_examples):.2f} calls/video")
print(f"  Total:               {sum(total_tool_usage.values()) / len(val_examples):.2f} calls/video")

# Count how many videos used each tool
videos_with_analyze_video = sum(1 for r in results if r.get('tool_usage', {}).get('analyze_video', 0) > 0)
videos_with_pose_data = sum(1 for r in results if r.get('tool_usage', {}).get('extract_pose_data', 0) > 0)
videos_with_both = sum(1 for r in results if r.get('tool_usage', {}).get('analyze_video', 0) > 0 and r.get('tool_usage', {}).get('extract_pose_data', 0) > 0)

print(f"\nTool usage distribution:")
print(f"  Videos using analyze_video:      {videos_with_analyze_video} ({videos_with_analyze_video/len(val_examples)*100:.1f}%)")
print(f"  Videos using extract_pose_data:  {videos_with_pose_data} ({videos_with_pose_data/len(val_examples)*100:.1f}%)")
print(f"  Videos using both tools:          {videos_with_both} ({videos_with_both/len(val_examples)*100:.1f}%)")



Per-Strike Accuracy:
------------------------------------------------------------
  cross               : 15.79% (3/19 correct, 7 predicted)
  jab                 : 31.58% (6/19 correct, 28 predicted)
  knee                : 84.21% (16/19 correct, 42 predicted)
  lead_elbow          :  0.00% (0/19 correct, 1 predicted)
  lead_hook           :  0.00% (0/19 correct, 1 predicted)
  lead_uppercut       :  0.00% (0/19 correct, 0 predicted)
  low_kick            :  0.00% (0/19 correct, 0 predicted)
  rear_elbow          :  0.00% (0/19 correct, 1 predicted)
  rear_hook           :  0.00% (0/19 correct, 3 predicted)
  rear_uppercut       :  0.00% (0/19 correct, 0 predicted)
  roundhouse          : 15.79% (3/19 correct, 58 predicted)
  switch_kick         :  0.00% (0/19 correct, 6 predicted)
  switch_knee         :  0.00% (0/19 correct, 1 predicted)
  teep                : 57.89% (11/19 correct, 115 predicted)
  the_video_shows_a_lead_hook.:  0.00% (0/0 correct, 1 predicted)
  the_video_shows_

## Section 5: Results and Comparison


In [9]:
# Save evaluation results (same format as DSPy and baseline notebooks)
output_dir = Path("results")
output_dir.mkdir(exist_ok=True)

# Save evaluation results
results_data = {
    'metrics': {
        'exact_match_accuracy': exact_match_accuracy,
        'total_examples': len(val_examples),
        'exact_matches': exact_matches
    },
    'tool_usage': {
        'total_calls': total_tool_usage,
        'average_calls_per_video': {
            'analyze_video': total_tool_usage['analyze_video'] / len(val_examples) if val_examples else 0,
            'extract_pose_data': total_tool_usage['extract_pose_data'] / len(val_examples) if val_examples else 0,
            'total': sum(total_tool_usage.values()) / len(val_examples) if val_examples else 0
        },
        'videos_using_analyze_video': videos_with_analyze_video,
        'videos_using_extract_pose_data': videos_with_pose_data,
        'videos_using_both_tools': videos_with_both
    },
    'results': results,
    'strike_stats': {k: dict(v) for k, v in strike_stats.items()}
}

with open(output_dir / "individual_agent_evaluation_results.json", 'w') as f:
    json.dump(results_data, f, indent=2)

print(f"Results saved to {output_dir / 'individual_agent_evaluation_results.json'}")


Results saved to results\individual_agent_evaluation_results.json


In [10]:
# Display summary
print("\n" + "=" * 60)
print("AGENT EVALUATION SUMMARY")
print("=" * 60)
print(f"Model: Gemini 2.5 Flash (LangGraph Agent)")
print(f"Approach: Agent-based with tools (analyze_video, extract_pose_data)")
print(f"Training examples: {len(train_examples)}")
print(f"Validation examples: {len(val_examples)}")
print(f"\nFinal Performance:")
print(f"  Exact Match Accuracy: {exact_match_accuracy:.2%}")
print(f"\nTool Usage:")
print(f"  Total analyze_video calls:      {total_tool_usage['analyze_video']}")
print(f"  Total extract_pose_data calls:  {total_tool_usage['extract_pose_data']}")
print(f"  Average calls per video:       {sum(total_tool_usage.values()) / len(val_examples):.2f}")
print(f"\nResults saved to: {output_dir}/individual_agent_evaluation_results.json")
print("\nNote: Compare these results with DSPy and baseline classifier results for evaluation.")
print("=" * 60)



AGENT EVALUATION SUMMARY
Model: Gemini 2.5 Flash (LangGraph Agent)
Approach: Agent-based with tools (analyze_video, extract_pose_data)
Training examples: 70
Validation examples: 266

Final Performance:
  Exact Match Accuracy: 14.66%

Tool Usage:
  Total analyze_video calls:      532
  Total extract_pose_data calls:  530
  Average calls per video:       3.99

Results saved to: results/individual_agent_evaluation_results.json

Note: Compare these results with DSPy and baseline classifier results for evaluation.
